# Step 5: The transformer block

<!-- notebook-only -->

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell. An even nicer way to read: the website, built with `.venv/bin/python website/build.py --serve`.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They recompute every table and let you experiment. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("5.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.

<!-- /notebook-only -->

**Goal:** build the whole "brain" of the toddler: attention, then a small layer of neurons where each token thinks on its own, with every part *adding* its result to the token's vector. At the end of the block, each vector looks like the token that should come next.

In [ ]:
# Run this first (practice only): imports, our sentence (Steps 1 to 3) and attention (Step 4)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from toddler.attention import STREAM, attention, hint_attention
from toddler.bpe import BPETokenizer
from toddler.data import CORPUS
from toddler.embeddings import toddler_table
from toddler.exercises import check, solution
from toddler.positions import with_positions


def short(x):
    return "0" if abs(x) < 1e-9 else f"{x:.2f}".rstrip("0").rstrip(".")


np.set_printoptions(formatter={"float_kind": short})
pd.set_option("display.float_format", short)

tok = BPETokenizer.train(CORPUS)
table = toddler_table(tok)                   # the embedding table (Step 2): one row of 8 numbers per token
ids = tok.encode("I am hungry . I want milk .")
labels = [f"#{p} {tok.vocab[i]}" for p, i in enumerate(ids, start=1)]
inputs = with_positions(table[ids])          # X: 8 rows of 10 numbers (Step 3)
heads = hint_attention()
added, weights = attention(inputs, **heads)  # what attention adds to each row (Step 4)
print("ready: 8 input vectors of", len(STREAM), "numbers, and what attention adds to each of them")

## 5.1 The residual stream: add, never replace

In Step 4, attention did not *replace* `_want`'s vector: it **added** its result to it. The same rule holds for every part of the block, and in every real model:

**vector = vector + (what the part computed)**

Picture the vector as a notebook passed along a row of students. Each student reads it and writes a note into it. Nobody tears out a page: to cancel something, a student writes a negative number.

This path, where each token's vector flows from part to part and collects their notes, is called the **residual stream**. Adding instead of replacing has three big advantages:
- The token never forgets what it is: `_want` still has want = 1 after attention.
- A part with nothing useful to say adds zeros, and nothing is lost.
- During training (Step 7), the pushes flow back through the "+" unchanged, which makes even very deep models trainable.

After attention, the stream of our sentence is X + mix · Wo (Step 4, section 4.7). We call it **S**, for stream. It is still an 8 × 10 matrix, one row per token:

```
              me   be want  stop feel  food   hug sleep    px    py
#1 _I          1    0    0     0    0     0     0     0     1     0
#2 _am         0    1    0     0    0     0     0     0  0.71  0.71
#3 _hungry     0    0    0     0    1  1.33     0     0     0     1
#4 _.          0    0    0     1    0  0.25     0     0 -0.71  0.71
#5 _I          1    0    0     0    0   0.2     0     0    -1     0
#6 _want       0    0    1     0    0  0.99     0     0 -0.71 -0.71
#7 _milk       0    0    0     0    0  1.29     0     0     0    -1
#8 _.          0    0    0     1    0  0.25     0     0  0.71 -0.71
```

The small bits of `food` in rows 3, 4, 5, 7 and 8 come from the tokens that spread their attention evenly (Step 4, section 4.7). Row 6, `_want`, now carries 0.99 food.

In [ ]:
# Try it: the stream after attention = the input vectors + what attention adds
stream = inputs + added
pd.DataFrame(stream, index=labels, columns=STREAM)

## 5.2 One neuron, step by step

After attention, each token goes through a layer of **neurons**. A neuron is a tiny detector. It looks at the 10 numbers of **one** token and answers one yes/no question, such as "is this `_I`, with the clock pointing right?". Its answer is **one number**, its output y: 0 means "no" (the neuron is **silent**), and a positive number means "yes" (the neuron **fires**).

In textbooks, a neuron is written **y = f(w₁x₁ + w₂x₂ + … + w₁₀x₁₀ + b)**:
- x₁ … x₁₀ are its inputs: the 10 numbers of the token;
- w₁ … w₁₀ are its **weights**, one per input. We call them its **read column**;
- b is its **bias**, one number;
- f is its **activation function**: for us, **ReLU**;
- y is its output: one number.

Let's follow one neuron of our toddler, "I, at the start", on the first `_I` (#1, row 1 of S). Steps 1 and 2 happen inside the neuron. Steps 3 and 4 happen after it: they show where its output y goes.

**1. Read: a dot product, plus the bias.** The read column says how much each of the 10 numbers counts: a positive number is evidence **for** a yes, a negative number is evidence **against**, and 0 means "ignored". This neuron reads `me` and `px`, each × 1, and ignores the 8 other columns:

```
                    me   be want stop feel food  hug sleep    px    py
_I (#1) in S         1    0    0    0    0    0    0     0     1     0
read column          1    0    0    0    0    0    0     0     1     0
multiply             1    0    0    0    0    0    0     0     1     0   → sum = 2
```

Then it adds its **bias**, here −1: 2 + (−1) = **1**. This is the inside of the formula, w₁x₁ + … + w₁₀x₁₀ + b, often called **z**: here z = 1. The bias is a threshold: with −1, the neuron needs more than 1 of evidence to fire. `me` alone would give 1 − 1 = 0, not enough.

**2. ReLU: fire, or stay silent.** **ReLU** keeps z if it is positive, and gives 0 otherwise: ReLU(z) = the bigger of z and 0.

![ReLU: below 0 the neuron is silent, above 0 it keeps the number](figures/05_relu.png)

| z | −2 | −0.29 | 0 | 0.5 | 1 | 2 |
|---|---|---|---|---|---|---|
| ReLU(z) | 0 | 0 | 0 | 0.5 | 1 | 2 |

Here y = ReLU(1) = **1**: the neuron fires. That's the whole neuron: **y = ReLU(read column · x + bias)**, one number. The neuron itself doesn't write anything into the token.

**3. Layer 2, the output units: where y goes.** The output y goes to a **second layer**: **10 output units**, one for each column of the stream (`me` … `py`). The neuron is linked to each of them, and each link has its own weight. The 10 weights on the links that leave a neuron are its **write row**. For neuron 1: −1 toward `me`, +1 toward `be`, and 0 toward the 8 others.

Each output unit adds up what arrives on its links: the output y of each of the 40 neurons, times the weight of its link, plus the unit's own bias (0 in our hint). It's the formula above again, but without ReLU. Here only neuron 1 fires; the 39 others give y = 0:

| output unit | weight of its link from neuron 1 | × y = 1 | from the 39 other neurons | its output |
|---|---|---|---|---|
| `me` | −1 | −1 | 0 | **−1** |
| `be` | +1 | 1 | 0 | **1** |
| the 8 others | 0 | 0 | 0 | 0 |

These output units belong to the feed-forward layer. They are not the output layer of the whole model, which comes in Step 6.

**4. Add: into the token.** The 10 outputs are added to the token's 10 numbers (the residual stream, 5.1):

```
                    me   be want stop feel food  hug sleep    px    py
_I (#1) in S         1    0    0    0    0    0    0     0     1     0
+ outputs           -1    1    0    0    0    0    0     0     0     0
= new _I             0    1    0    0    0    0    0     0     1     0
```

The new vector has be = 1 and no more `me`: it looks like `_am`, the token that comes next. So the neuron never writes into the token: it only computes y, how strongly it fires. Its write row, the weights of its links, decides what layer 2 adds. The whole path in one picture:

![One neuron at the first _I: layer 1 (the neuron: read, ReLU), layer 2 (the 10 output units), then the add](figures/05_one_neuron.png)

In [ ]:
# Try it: one neuron, "I, at the start", on the first _I (row 1 of S), then layer 2 and the add
def relu(x):
    return np.maximum(x, 0)


read_column, write_row = np.zeros(len(STREAM)), np.zeros(len(STREAM))
read_column[STREAM.index("me")], read_column[STREAM.index("px")] = 1, 1  # the neuron's weights
write_row[STREAM.index("me")], write_row[STREAM.index("be")] = -1, 1    # its links to the 10 output units
bias = -1

first_I = stream[0]                          # row 1 is index 0: Python counts from 0
z = first_I @ read_column + bias             # 1. read: a dot product, plus the bias
y = relu(z)                                  # 2. ReLU: the neuron's output, one number
outputs = y * write_row                      # 3. layer 2: each output unit gets y × the weight of its link
print("1. read: z =", short(first_I @ read_column), "+ bias", bias, "=", short(z))
print("2. ReLU: y =", short(y), "→ it fires" if y > 0 else "→ silent")
pd.DataFrame([first_I, outputs, first_I + outputs],      # 4. add the outputs to the token
             index=["_I (#1) in S", "+ outputs (layer 2)", "= new _I"], columns=STREAM)

## 5.3 The feed-forward layer: 40 neurons, then 10 output units

Our toddler has **40 neurons** side by side, and every token goes through all 40 of them. Each token goes **on its own**: no token looks at another one here. Looking at the other tokens was attention's job. Now each token "thinks" about what it has collected.

The 40 neurons are **layer 1**. They all send their output y to the same 10 output units: **layer 2**. Like in attention, we stack the numbers into matrices:
- Layer 1: put the 40 read columns side by side: **W1**, a 10 × 40 matrix. One row per column of the stream (`me` … `py`), one column per neuron. The 40 biases make **b1**: 40 numbers, one per neuron.
- Layer 2: put the 40 write rows one under the other: **W2**, a 40 × 10 matrix. Row n holds the 10 links that leave neuron n, and column j holds the 40 links that arrive at output unit j. The 10 biases of the output units make **b2**; our hint keeps them at 0.

This is built like Wv and Wo in attention: W1 goes from the 10 columns of the stream to 40 neurons, and W2 goes back from 40 to 10. In between come the bias and ReLU. With the rule of Step 4 (each number of a result = a row of the left matrix · a column of the right one), the whole sentence goes through both layers in four moves:

```
S (8 × 10) · W1 (10 × 40) + b1  =  reads (8 × 40)   layer 1: what each neuron reads
ReLU(reads)                     =  H (8 × 40)       layer 1: the output y of each neuron
H (8 × 40) · W2 (40 × 10) + b2  =  F (8 × 10)       layer 2: the 10 outputs, for each token
S + F                           =  the final vectors (8 × 10)
```

In reads and H, row t and column n are about token t and neuron n. In one line: **F = ReLU(S · W1 + b1) · W2 + b2**. Together, the two layers are called the **feed-forward** layer, or **MLP** (multi-layer perceptron): the numbers only go forward, from the stream to the neurons, to the output units, and back into the stream.

**Why 40 neurons?** Like the 4 slots of attention, it's a design choice. Real models usually take 4 times the vector size: 4 × 10 = 40. More neurons can detect more patterns, but they cost more numbers: W1 and W2 already have 400 numbers each. Our hint only needs 7 neurons. The other 33 are all zeros: they read 0, plus a bias of 0, so their output y is always 0, and their links to layer 2 are 0 too.

Here are the two layers as a picture, with the 7 neurons of our hint. Each line on the left is a number of W1 that is not 0: a weight of a neuron. Each line on the right is a number of W2 that is not 0: a link from a neuron to an output unit.

![The feed-forward layer: the token's 10 numbers, layer 1 (the neurons, 7 of 40 used), layer 2 (the 10 output units)](figures/05_feed_forward_layer.png)

## 5.4 Our 7 hint neurons, one by one

Training will set all 40 neurons (Step 7). For our hint, we set 7 of them by hand. Each one detects one kind of token. Its write row is **fixed in advance**: we chose it once, when we built the neuron, as **the token that usually comes next − this token**. When the neuron runs, it only sees the current token, never the next one: when the toddler writes, the next token doesn't even exist yet. The neuron only computes y; its fixed write row decides what layer 2 adds, and the result looks like the token that usually comes next. We know what usually comes next because we know the toddler's 3 sentences; training learns it the same way, from the training text (Step 7).

For example, in all 3 sentences, `_I` at the start is followed by `_am`:
- when we built neuron 1, we chose its write row: `_am` − `_I` = (be 1) − (me 1) = **− me + be**;
- later, each time it fires on `_I` at the start (y = 1), layer 2 gives that same row, × 1, and it is added to `_I`: `_I` + (− me + be) = be 1, which looks like `_am`.

When the next token can vary (after `_am` or `_want`), the recipe is adapted: see the tricks below the table.

| # | neuron | reads (column of W1, then the bias) | write row (row of W2) | turns it into |
|---|---|---|---|---|
| 1 | I, at the start | me + px − 1 | − me + be | `_am` |
| 2 | I, later | me − px − 1 | − me + want | `_want` |
| 3 | am | be | − be + feel + ⅓ food + ⅓ hug + ⅓ sleep | any feeling |
| 4 | a feeling | feel | − feel − food − hug − sleep + stop | `_.` |
| 5 | the stop | stop | − stop + me | `_I` |
| 6 | want | want | − want − feel | a thing |
| 7 | a thing | food + hug + sleep − 2 × (me + be + want + stop + feel) | − food − hug − sleep + stop | `_.` |

The − 1 of neurons 1 and 2 is their bias; the 5 others have a bias of 0. Four tricks are worth a closer look:
- **The bias as a threshold (neurons 1 and 2).** `me` alone gives 1 − 1 = 0: silent. Neuron 1 also needs px = 1, the clock pointing right (#1), and neuron 2 needs px = −1, the clock pointing left (#5). So each fires only for `me` **and** the right position: an **AND**. Section 5.5 works it out.
- **Not knowing yet (neuron 3).** After `_am` comes a feeling, but `_am` can't know which one. So its write row has ⅓ of each topic: "a feeling, any of the 3".
- **"Surely not" (neuron 6).** After attention, `_want` carries food, like `_milk` (food), but also like `_hungry` (feel + food). The − feel of its write row says "surely not a feeling", so the vector ends up closer to `_milk` than to `_hungry` (section 5.7).
- **A veto (neuron 7).** It fires for the things (`_milk`, `_mommy`, `_bed`). It reads the 3 topics, but each small word and `feel` count −2: a veto, big enough to beat any topic. `_milk` has food 1.29 and nothing else, so it reads 1.29 and fires. `_hungry` has food 1.33, but also feel 1: 1.33 − 2 = −0.67, silent. Only a pure topic, a thing, makes it fire.

Neuron 4 removes all three topics, whatever the feeling. For `_hungry` (food 1.33 after attention), that leaves food 0.33, hug −1 and sleep −1. These leftovers are harmless: the vector still looks like `_.` most of all (section 5.8).

The same 7 neurons as matrices. **W1**, its first 7 columns (n1 to n7 are the neurons of the table; the 33 other columns are 0), with **b1** in the last row:

```
            n1   n2   n3   n4   n5   n6   n7
me           1    1    0    0    0    0   -2
be           0    0    1    0    0    0   -2
want         0    0    0    0    0    1   -2
stop         0    0    0    0    1    0   -2
feel         0    0    0    1    0    0   -2
food         0    0    0    0    0    0    1
hug          0    0    0    0    0    0    1
sleep        0    0    0    0    0    0    1
px           1   -1    0    0    0    0    0
py           0    0    0    0    0    0    0
bias b1     -1   -1    0    0    0    0    0
```

**W2**, its first 7 rows: row n holds the links from neuron n to the 10 output units (the 33 other rows are 0):

```
        me   be want  stop feel  food   hug sleep    px    py
n1      -1    1    0     0    0     0     0     0     0     0
n2      -1    0    1     0    0     0     0     0     0     0
n3       0   -1    0     0    1  0.33  0.33  0.33     0     0
n4       0    0    0     1   -1    -1    -1    -1     0     0
n5       1    0    0    -1    0     0     0     0     0     0
n6       0    0   -1     0   -1     0     0     0     0     0
n7       0    0    0     1    0    -1    -1    -1     0     0
```

Read a neuron's column of W1 from top to bottom, and its row of W2 from left to right: they are the "reads" and the "write rows" of the table above.

In [ ]:
# Try it: the 7 hint neurons as matrices: W1 (10 × 40) with b1 (40 numbers), and W2 (40 × 10) with b2 (10 numbers)
HIDDEN = 40
NEURONS = [  # (name, what it reads, bias, its write row)
    ("I, at the start", {"me": 1, "px": 1}, -1, {"me": -1, "be": 1}),
    ("I, later", {"me": 1, "px": -1}, -1, {"me": -1, "want": 1}),
    ("am", {"be": 1}, 0, {"be": -1, "feel": 1, "food": 1 / 3, "hug": 1 / 3, "sleep": 1 / 3}),
    ("a feeling", {"feel": 1}, 0, {"feel": -1, "food": -1, "hug": -1, "sleep": -1, "stop": 1}),
    ("the stop", {"stop": 1}, 0, {"stop": -1, "me": 1}),
    ("want", {"want": 1}, 0, {"want": -1, "feel": -1}),
    ("a thing", {"food": 1, "hug": 1, "sleep": 1, "me": -2, "be": -2, "want": -2, "stop": -2, "feel": -2}, 0,
     {"food": -1, "hug": -1, "sleep": -1, "stop": 1}),
]
W1, b1 = np.zeros((len(STREAM), HIDDEN)), np.zeros(HIDDEN)
W2, b2 = np.zeros((HIDDEN, len(STREAM))), np.zeros(len(STREAM))
for n, (_, reads_from, neuron_bias, writes_to) in enumerate(NEURONS):
    for name, weight in reads_from.items():
        W1[STREAM.index(name), n] = weight   # neuron n reads with column n of W1
    b1[n] = neuron_bias
    for name, weight in writes_to.items():
        W2[n, STREAM.index(name)] = weight   # its links to the output units are row n of W2

names = [name for name, *_ in NEURONS]
used = [f"n{n}" for n in range(1, len(NEURONS) + 1)]
print("W1 (first 7 columns of 40), with b1 in the last row:")
print(pd.DataFrame(np.vstack([W1[:, :7], b1[:7]]), index=STREAM + ["bias b1"], columns=used))
print("\nW2 (first 7 rows of 40):")
pd.DataFrame(W2[:7], index=used, columns=STREAM)

## 5.5 Worked example: the two `_I`, at last told apart

Rows 1 and 5 of S are both `_I`. They differ only in their position (and a little food from attention): px = 1 at #1, and px = −1 at #5. Neuron 1 reads me + px − 1, and neuron 2 reads me − px − 1:

| | `_I` at #1: me = 1, px = 1 | `_I` at #5: me = 1, px = −1 |
|---|---|---|
| neuron 1 reads | 1 + 1 − 1 = **1** → ReLU → **1: fires** | 1 + (−1) − 1 = −1 → ReLU → 0: silent |
| neuron 2 reads | 1 − 1 − 1 = −1 → ReLU → 0: silent | 1 − (−1) − 1 = **1** → ReLU → **1: fires** |

So layer 2 gives the first `_I` the write row of neuron 1, "− me + be", and it looks like `_am`; it gives the second `_I` the write row of neuron 2, "− me + want", and it looks like `_want`. This is where the clock of Step 3 pays off: the same token makes different neurons fire at different places. The − 1 (the bias) makes each neuron an **AND**: it only fires when me **and** the right clock direction are both there.

In [ ]:
# Try it: neurons 1 and 2 at the two _I (rows 1 and 5 of S), before and after ReLU
reads_I = stream[[0, 4]] @ W1[:, :2] + b1[:2]
pd.DataFrame(np.hstack([reads_I, relu(reads_I)]), index=["_I at #1", "_I at #5"],
             columns=["n1 reads", "n2 reads", "n1 after ReLU", "n2 after ReLU"])

## 5.6 Why ReLU matters: the first `_I` without it

In 5.5, the silent neurons stayed silent thanks to ReLU. What if we removed it, and kept the raw reads? At the first `_I`, three neurons read something that is not 0:

| neuron | reads | with ReLU | without ReLU |
|---|---|---|---|
| 1 · I, at the start | me + px − 1 = 1 + 1 − 1 = 1 | 1: fires | 1 |
| 2 · I, later | me − px − 1 = 1 − 1 − 1 = −1 | 0: silent | −1 |
| 7 · a thing | 0 − 2 × me = −2 | 0: silent | −2 |

Without ReLU, a neuron below its threshold is not silent: its output y is negative, so layer 2 gets its write row **backwards**. Neuron 2 gives y = −1, so layer 2 adds −1 × (− me + want) = + me − want. Neuron 7 gives y = −2: −2 × (− food − hug − sleep + stop) = + 2 food + 2 hug + 2 sleep − 2 stop:

```
                       me   be want stop feel food  hug sleep    px    py
_I (#1) in S            1    0    0    0    0    0    0     0     1     0
n1:  1 × write row     -1    1    0    0    0    0    0     0     0     0
n2: -1 × write row      1    0   -1    0    0    0    0     0     0     0
n7: -2 × write row      0    0    0   -2    0    2    2     2     0     0
= without ReLU          1    1   -1   -2    0    2    2     2     1     0
= with ReLU             0    1    0    0    0    0    0     0     1     0
```

With ReLU, the new `_I` is be 1 (and px 1): it looks like `_am`. Without ReLU, it is a mess. Its dot product (Step 2) with each of `_hungry`, `_sad`, `_tired`, `_milk`, `_mommy` and `_bed` is 2, and only 1 with `_am`.

There is a deeper reason. Without ReLU, the layer computes (S · W1 + b1) · W2 = S · (W1 · W2) + b1 · W2: the two matrices collapse into one single 10 × 10 matrix, W1 · W2. One matrix applies the same fixed recipe to every token, "a bit of this column plus a bit of that one". It can never say "**only if**": only if `me` **and** the clock points right. ReLU is what lets each neuron make a yes/no decision.

In [ ]:
# Try it: the first _I with and without ReLU, and which token each version looks like
WORDS = ["_I", "_am", "_hungry", "_sad", "_tired", "_.", "_want", "_milk", "_mommy", "_bed"]


def looks_like(vector):
    # the dot product (Step 2) of the 8 meaning numbers with the embedding row of each token of our sentences
    return pd.Series({word: vector[:8] @ table[tok.token_to_id[word]] for word in WORDS})


reads = stream @ W1 + b1                     # what every neuron reads at every token: 8 × 40
with_relu = stream + relu(reads) @ W2 + b2
without_relu = stream + reads @ W2 + b2      # the same, but the negative reads are kept
print(pd.DataFrame([with_relu[0], without_relu[0]], index=["with ReLU", "without ReLU"], columns=STREAM))
pd.DataFrame({"with ReLU": looks_like(with_relu[0]), "without ReLU": looks_like(without_relu[0])}).T

## 5.7 Worked example: `_want` through the two layers

After attention, `_want` (row 6 of S) is: want = 1, food = 0.99, px = −0.71, py = −0.71, and 0 elsewhere. Row 6 of S · W1 + b1 gives the 40 reads of `_want`. The 33 unused neurons read 0; here are the 7 others:

| # | neuron | reads | ReLU |
|---|---|---|---|
| 1 | I, at the start | me + px − 1 = 0 + (−0.71) − 1 = −1.71 | 0 |
| 2 | I, later | me − px − 1 = 0 + 0.71 − 1 = −0.29 | 0 |
| 3 | am | be = 0 | 0 |
| 4 | a feeling | feel = 0 | 0 |
| 5 | the stop | stop = 0 | 0 |
| 6 | want | want = 1 | **1** |
| 7 | a thing | 0.99 − 2 × (0 + 0 + 1 + 0 + 0) = −1.01 | 0 |

Only neuron 6 fires, with y = 1. So row 6 of H is (0, 0, 0, 0, 0, 1, 0, 0, …, 0), and layer 2 gives row 6 of H · W2 = 1 × (row 6 of W2) = − want − feel: the output units `want` and `feel` get −1, and the 8 others get 0. In the picture, a colored line carries a number that is not 0 (blue: a + number, orange: a − number), and a gray line carries 0:

![_want through the two layers: only neuron 6 fires, and layer 2 gives − want − feel](figures/05_want_through_layer.png)

Then − want − feel is added to `_want`:

![_want through the block: attention adds food, the feed-forward layer removes want and rules out the feelings](figures/05_want_through_block.png)

**Does it look like `_milk`?** "Looks like" means: has a big dot product (Step 2) with that token's row of the embedding table, using the 8 meaning numbers. For the final `_want`:

| final `_want` · | `_milk` | `_mommy` | `_bed` | `_hungry` | `_want` |
|---|---|---|---|---|---|
| dot product | **0.99** | 0 | 0 | −0.01 | 0 |

`_milk` wins. Without the − feel of neuron 6, `_hungry` would tie with it: both would get 0.99.

In [ ]:
# Try it: _want (row 6 of S) through the two layers: its 7 reads and ReLU (layer 1), then what layer 2 gives
print(pd.DataFrame({"reads": reads[5, :7], "after ReLU": relu(reads[5, :7])}, index=names))
thinking = relu(reads[5]) @ W2 + b2
final_want = stream[5] + thinking
pd.DataFrame([stream[5], thinking, final_want], index=["_want in S", "+ feed-forward", "= final _want"],
             columns=STREAM)

In [ ]:
# Try it: which token does the final _want look like?
looks_like(final_want)

## 5.8 The whole block, for every token at once

All 8 tokens go through the same 40 neurons at the same time: that's the four matrix moves of 5.3. First the reads, S · W1 + b1, and H = ReLU(reads), in their first 7 columns:

```
           reads = S · W1 + b1                         H = ReLU(reads)
               n1    n2    n3    n4    n5    n6    n7     n1   n2   n3   n4   n5   n6   n7
#1 _I           1    -1     0     0     0     0    -2      1    0    0    0    0    0    0
#2 _am      -0.29 -1.71     1     0     0     0    -2      0    0    1    0    0    0    0
#3 _hungry     -1    -1     0     1     0     0 -0.67      0    0    0    1    0    0    0
#4 _.       -1.71 -0.29     0     0     1     0 -1.75      0    0    0    0    1    0    0
#5 _I          -1     1     0     0     0     0  -1.8      0    1    0    0    0    0    0
#6 _want    -1.71 -0.29     0     0     0     1 -1.01      0    0    0    0    0    1    0
#7 _milk       -1    -1     0     0     0     0  1.29      0    0    0    0    0    0 1.29
#8 _.       -0.29 -1.71     0     0     1     0 -1.75      0    0    0    0    1    0    0
```

ReLU turns every negative read into 0. Each token makes **exactly one** neuron fire: #1 `_I` neuron 1, #2 `_am` neuron 3, #3 `_hungry` neuron 4, #4 `_.` neuron 5, #5 `_I` neuron 2, #6 `_want` neuron 6, #7 `_milk` neuron 7 (with y = 1.29, its food after attention), and #8 `_.` neuron 5 again. Then layer 2 gives F = H · W2 + b2 (8 × 10), what is added to each token:

```
              me   be want  stop feel  food   hug sleep    px    py
#1 _I         -1    1    0     0    0     0     0     0     0     0
#2 _am         0   -1    0     0    1  0.33  0.33  0.33     0     0
#3 _hungry     0    0    0     1   -1    -1    -1    -1     0     0
#4 _.          1    0    0    -1    0     0     0     0     0     0
#5 _I         -1    0    1     0    0     0     0     0     0     0
#6 _want       0    0   -1     0   -1     0     0     0     0     0
#7 _milk       0    0    0  1.29    0 -1.29 -1.29 -1.29     0     0
#8 _.          1    0    0    -1    0     0     0     0     0     0
```

Here is the whole block: attention, then the feed-forward layer, each one adding its result to the stream:

![One transformer block](figures/05_block.png)

The final vectors, S + F (8 × 10), and the token each one looks like (the biggest dot product, as in 5.7):

| pos | token | final vector (the numbers that are not 0) | looks like |
|---|---|---|---|
| 1 | `_I` | be 1, px 1 | `_am` |
| 2 | `_am` | feel 1, food 0.33, hug 0.33, sleep 0.33, px 0.71, py 0.71 | any feeling: `_hungry`, `_sad` or `_tired` (a tie) |
| 3 | `_hungry` | stop 1, food 0.33, hug −1, sleep −1, py 1 | `_.` |
| 4 | `_.` | me 1, food 0.25, px −0.71, py 0.71 | `_I` |
| 5 | `_I` | want 1, food 0.2, px −1 | `_want` |
| 6 | `_want` | feel −1, food 0.99, px −0.71, py −0.71 | `_milk` |
| 7 | `_milk` | stop 1.29, hug −1.29, sleep −1.29, py −1 | `_.` |
| 8 | `_.` | me 1, food 0.25, px 0.71, py −0.71 | `_I` |

Each one looks like **the token that comes next**. That is the whole job of the transformer: to turn each token's vector into a good description of the next token. Step 6 reads these final vectors and turns them into probabilities.

In [ ]:
# Try it: the whole block for the whole sentence
def feed_forward(stream, W1, b1, W2, b2):
    return relu(stream @ W1 + b1) @ W2 + b2  # F = ReLU(S · W1 + b1) · W2 + b2


def block(inputs, params):
    added, _ = attention(inputs, params["Wq"], params["Wk"], params["Wv"], params["Wo"])
    stream = inputs + added                  # S = X + what attention adds
    return stream + feed_forward(stream, params["W1"], params["b1"], params["W2"], params["b2"])  # S + F


params = {**heads, "W1": W1, "b1": b1, "W2": W2, "b2": b2}
final = block(inputs, params)
print("H = ReLU(S · W1 + b1), first 7 columns of 40:")
print(pd.DataFrame(relu(reads[:, :7]), index=labels, columns=used))
pd.DataFrame(final, index=labels, columns=STREAM)

In [ ]:
# Try it: which token each final vector looks like (the biggest dot product)
dots = pd.DataFrame({label: looks_like(vector) for label, vector in zip(labels, final)}).T
dots["looks like"] = [" or ".join(row.index[np.isclose(row, row.max())]) for _, row in dots.iterrows()]
dots

## 5.9 Normalization: keeping the numbers steady

Real models add one more move before attention and before the feed-forward layer: **layer normalization**. Each vector is rescaled so that its numbers have an average of 0 and a typical size of 1. Then two learned tables, a scale and a shift, adjust it.

Worked example on a small vector (2, 4, 6, 8):
1. The average is (2 + 4 + 6 + 8) ÷ 4 = 5. Subtract it: (−3, −1, 1, 3).
2. The spread is √((9 + 1 + 1 + 9) ÷ 4) = √5 = 2.24. Divide by it: **(−1.34, −0.45, 0.45, 1.34)**.

Why? With dozens of blocks all adding to the stream, the numbers could slowly grow huge or shrink to nothing. Normalizing gives each part a steady input. Our toddler has a single block and small numbers, so it skips this move.

## 5.10 Stacking blocks

A transformer is several blocks in a row, each one with its own tables. Each block reads the stream and adds its notes, and the next block reads them.

| | toddler | GPT-2 small | GPT-3 |
|---|---|---|---|
| blocks | 1 | 12 | 96 |
| numbers per vector | 10 | 768 | 12,288 |
| neurons per feed-forward layer | 40 | 3,072 | 49,152 |

The feed-forward layer is usually 4 times wider than the vector (toddler: 10 → 40 → 10), and it holds about two thirds of a model's numbers. In our toddler's block, it has 850 numbers (W1 and W2: 400 each, b1: 40, b2: 10), against 160 for attention (Wq, Wk, Wv and Wo: 40 each). Studies suggest that much of what a model "knows" (that Paris is in France…) is stored in these neurons. Modern models use smoother versions of ReLU (GELU, SwiGLU), but the idea is the same.

## 5.11 What is still missing

The final vectors look like the next token, but the toddler must still **say** something: a probability for each of the 53 tokens. That is Step 6, output and softmax.

In [ ]:
# Try it: layer normalization of (2, 4, 6, 8)
v = np.array([2.0, 4, 6, 8])
centered = v - v.mean()
spread = np.sqrt((centered ** 2).mean())
print("average:", short(v.mean()), "  spread:", short(spread), "  normalized:", centered / spread)

<!-- notebook-only -->

The same code is saved in `toddler/block.py`, and the next notebooks import it from there.

<!-- /notebook-only -->

In [ ]:
# Try it: the package version used by the next notebooks gives the same final vectors
from toddler import block as toddler_block

package_final, _ = toddler_block.block(inputs, {**heads, **toddler_block.hint_feed_forward()})
assert np.allclose(package_final, final)
print("toddler/block.py gives the same final vectors")

## Your turn

Do each one **by hand first**, using the matrices and tables above.

### 5.1 What is ReLU of −2, of 0 and of 1.5?

<details>
<summary>Show the answer</summary>
<!-- answer: 0 0 1.5 -->
<!-- format: three numbers, e.g. 1 2 3 -->

ReLU keeps a number if it is positive, and gives 0 otherwise: ReLU(−2) = **0**, ReLU(0) = **0**, ReLU(1.5) = **1.5**.

</details>

In [ ]:
my_answer = None  # three numbers, e.g. [1, 2, 3]
check("5.1", my_answer)

### 5.2 Which hint neuron fires at `_am` (#2)?

<details>
<summary>Show the answer</summary>
<!-- answer: am | 3 | neuron 3 -->
<!-- format: the name of the neuron, e.g. want -->

The neuron **am** (number 3): it reads `be`, and `_am` has be = 1. Neuron 1 reads 0 + 0.71 − 1 = −0.29 and stays silent, and every other neuron reads 0 or less.

</details>

In [ ]:
my_answer = None  # the name of the neuron, e.g. "want"
check("5.2", my_answer)

### 5.3 What does the neuron "I, at the start" give at the second `_I` (#5)?

<details>
<summary>Show the answer</summary>
<!-- answer: 0 -->
<!-- format: one number -->

It reads me + px − 1 = 1 + (−1) − 1 = −1, and ReLU(−1) = **0**: it stays silent. The neuron "I, later" fires instead.

</details>

In [ ]:
my_answer = None  # one number
check("5.3", my_answer)

<!-- notebook-only -->

### 5.4 (code) Write `my_relu(x)`: it keeps each number if it is positive, and gives 0 otherwise.

<details>
<summary>Show the answer</summary>

```python
def my_relu(x):
    return np.maximum(x, 0)
```

</details>

<!-- /notebook-only -->

In [ ]:
def my_relu(x):
    # your code here (x is a NumPy array)
    return None


check("5.4", my_relu)

### 5.5 Normalize the vector (1, 3): subtract the average, then divide by the spread.

<details>
<summary>Show the answer</summary>
<!-- answer: -1 1 -->
<!-- format: two numbers, e.g. 0.5 -0.5 -->

1. The average is (1 + 3) ÷ 2 = 2. Subtract it: (−1, 1).
2. The spread is √(((−1)² + 1²) ÷ 2) = √1 = 1. Divide by it: **(−1, 1)**.

</details>

In [ ]:
my_answer = None  # two numbers, e.g. [0.5, -0.5]
check("5.5", my_answer)

### 5.6 Think: why does each part *add* its result to the vector, instead of replacing it?

<details>
<summary>Show the answer</summary>
<!-- choices: *The token keeps what it is, a part with nothing to say changes nothing, and the pushes of training flow straight back | Adding is faster to compute than replacing | Replacing would make the vectors longer | There is no real reason, both work the same -->

Adding keeps everything that is already in the vector: `_want` still knows it is `_want` after attention. A part with nothing useful to say adds zeros and loses nothing. And during training, the "+" passes the pushes straight back to the earlier parts, which is what makes deep models trainable.

</details>

### 5.7 Think: without ReLU, what would two matrices in a row (W1, then W2) be?

<details>
<summary>Show the answer</summary>
<!-- choices: *The same as one single matrix: the neurons could never make yes/no decisions | Twice as powerful as one matrix | Impossible to compute | A random matrix -->

Reading with W1 and writing with W2, without anything in between, is the same as using one single matrix: (S · W1) · W2 = S · (W1 · W2), a 10 × 10 matrix. ReLU breaks this: it lets a neuron be silent or fire, so it can make decisions like "`_I` **and** the clock points right". Section 5.6 shows what goes wrong at the first `_I` without it.

</details>

## Summary

- The **residual stream**: every part *adds* its result to the token's vector, it never replaces it. After attention, the stream is S = X + what attention adds (8 × 10).
- A **neuron** is a tiny detector: **y = ReLU(read column · x + bias)**, the textbook y = f(w₁x₁ + … + w₁₀x₁₀ + b) with f = ReLU. Its output y is one number, how strongly it fires: 0 means silent.
- The **feed-forward** layer has two layers, and each token goes through it on its own: 40 neurons (layer 1), then 10 **output units**, one per column of the stream (layer 2). Each output unit adds up the outputs y of the 40 neurons, each times the weight of its link, without ReLU. The 10 links that leave a neuron are its **write row**. In matrices: **F = ReLU(S · W1 + b1) · W2 + b2**, with W1 10 × 40 and W2 40 × 10. F is then added to the stream.
- Without ReLU, W1 and W2 would collapse into one single matrix, and no neuron could make a yes/no decision.
- Our 7 hint neurons have write rows chosen **in advance**, as "the token that usually comes next − this token": a neuron never sees the next token: it only computes y, and layer 2 adds its stored write row, times y. With the clock, neurons 1 and 2 finally tell the two `_I` apart.
- A **transformer block** = attention + feed-forward, each one adding to the stream. Real models normalize the vectors before each part, and stack many blocks.
- After the block, each final vector looks like the token that should come next: `_want`'s says "food, not a feeling", like `_milk`.
- Next: **Step 6, output and softmax**.